# Flux Variability Analysis (FVA) on TCA Cycle Model

This notebook loads the `TCA_cycle` model, applies specific flux constraints with a 10% range, and performs FVA.

In [ ]:
import cobra
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import flux_variability_analysis
import pandas as pd
import numpy as np
from scipy.stats.qmc import LatinHypercube
import os

In [ ]:
# Load the model
model_path = "../../models/TCA_cycle.json"
model = cobra.io.load_json_model(model_path)
print(f"Loaded model '{model.id}' with {len(model.reactions)} reactions and {len(model.metabolites)} metabolites.")

## Apply Flux Constraints

We fix the following fluxes with a range of +/- 10%:
- R16 (PYR_B): 0.0700
- R17 (AKG_B): 0.2300
- R18 (OAA_B): 0.1200
- R19 (VALX): 0.0500
- R20 (LYSX): 0.0300

In [ ]:
# R1 (PYR input) is the reference flux, fixed at 1000 (v1 = 1 in normalised units)
R1_REF = 1000.0
model.reactions.get_by_id("R1").bounds = (R1_REF, R1_REF)

# Scale the measured efflux targets by R1_REF so that the cobra model is
# consistent with the normalised flux_analysis.ipynb (where v1 = 1).
targets = {
    'R16': 0.0700 * R1_REF,   # PYR_B
    'R17': 0.2300 * R1_REF,   # AKG_B
    'R18': 0.1200 * R1_REF,   # OAA_B
    'R19': 0.0500 * R1_REF,   # VALX
    'R20': 0.0300 * R1_REF,   # LYSX
}

print(f"Reference flux R1 = {R1_REF}")
print("Applying constraints (Target ± 10%):")
for rxn_id, value in targets.items():
    delta = value * 0.10
    lb, ub = value - delta, value + delta
    model.reactions.get_by_id(rxn_id).bounds = (lb, ub)
    print(f"  {rxn_id}: Target={value:.1f}, Range=[{lb:.1f}, {ub:.1f}]")

## Perform Flux Variability Analysis

In [ ]:
pd.set_option('display.max_rows', None)
# Run FVA
# fraction_of_optimum=0.0 allows exploring the full feasible space given constraints
fva_result = flux_variability_analysis(model, fraction_of_optimum=0.0, processes=1)
print("\nFVA Results:")
print(fva_result)

In [ ]:
# FVA feasible ranges for the three free fluxes
fva_ranges = {
    'R2': (fva_result.loc['R2', 'minimum'], fva_result.loc['R2', 'maximum']),
    'R8': (fva_result.loc['R8', 'minimum'], fva_result.loc['R8', 'maximum']),
    'R9': (fva_result.loc['R9', 'minimum'], fva_result.loc['R9', 'maximum']),
}
print("FVA ranges (absolute fluxes):")
for rxn, (lb, ub) in fva_ranges.items():
    print(f"  {rxn}: [{lb:.4f}, {ub:.4f}]")

# Latin Hypercube Sampling — 30 space-filling combinations
sampler = LatinHypercube(d=3, seed=42)
lhs_unit = sampler.random(n=30)   # shape (30, 3) in [0, 1]

bounds = list(fva_ranges.values())
scaled = np.column_stack([
    lhs_unit[:, i] * (ub - lb) + lb
    for i, (lb, ub) in enumerate(bounds)
])

flux_combinations = pd.DataFrame(scaled, columns=['R2', 'R8', 'R9'])
flux_combinations.index = range(1, 31)
flux_combinations.index.name = 'Sample'

# Divide by 1000 to match the flux scale used in flux_analysis.ipynb (v1 = 1)
flux_combinations_scaled = flux_combinations / 1000

out_path = "flux_samples_R2_R8_R9.csv"
flux_combinations_scaled.to_csv(out_path)
print(f"\nSaved {len(flux_combinations_scaled)} samples (÷ 1000) → {out_path}")
print(flux_combinations_scaled.round(4))